In [1]:
from pathlib import Path
import subprocess
import sys

In [2]:
sys.executable

'd:\\study-on-agent\\.venv\\Scripts\\python.exe'

In [41]:
SCRIPT_PATH = ROOT / "dev_skills" / "read-file" / "main.py"

result = subprocess.run(
    [sys.executable, str(SCRIPT_PATH), *["--file", str(SCRIPT_PATH)]],
    capture_output=True,
    text=True,
    cwd=ROOT
)

In [42]:
print(result.stdout)

import argparse
import sys
from pathlib import Path

def get_args():
    parser = argparse.ArgumentParser()
    parser.add_argument(
        '-f', '--file', required=True, help='File to read', type=str,
    )
    return parser

def main():
    parser = get_args()
    args = parser.parse_args()
    path = Path(args.file)
    if not path.is_file():
        print(f"no such file: {path}", file=sys.stderr)
        sys.exit(1)  # nonzero exit -- run_script folds this into tool_result as a refusal/error, same as delete_file's guardrail would
    print(path.read_text(encoding="utf-8"))  # <-- this is the "return value"

if __name__=='__main__':
    main()




In [3]:
def run_script(directory:Path, script:str, args: list[str])->str:
    script_path = directory / script
    if not script_path.is_file():
        raise FileNotFoundError(f"no such script in skill: {script_path}")
    result = subprocess.run(
        [sys.executable, str(script_path), *(args or [])],
        capture_output=True,
        text=True,
        cwd=ROOT,  # anchor relative path args at the repo root, always --
                   # never the notebook's own cwd, never the invoked script's folder
    )
    output = result.stdout
    if result.returncode != 0:
        # fold a guardrail refusal / error into the return value instead of
        # discarding it -- e.g. read_file's sys.exit(1) + stderr message
        output += f"\n[exit {result.returncode}]\n{result.stderr}"
    return output

In [4]:
Path.cwd().resolve()

WindowsPath('D:/study-on-agent/notebooks')

In [5]:
ROOT = Path.cwd().parent
SKILL_DIR = ROOT / "dev_skills"

In [8]:
list(SKILL_DIR.glob("*/SKILL.md"))

[WindowsPath('d:/study-on-agent/dev_skills/farewell/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/greeting/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/read-file/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/tell-joke/SKILL.md')]

In [9]:
list(SKILL_DIR.glob("**/*.md"))

[WindowsPath('d:/study-on-agent/dev_skills/farewell/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/greeting/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/read-file/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/tell-joke/SKILL.md'),
 WindowsPath('d:/study-on-agent/dev_skills/tell-joke/references/dad-jokes.md'),
 WindowsPath('d:/study-on-agent/dev_skills/tell-joke/references/knock-knock.md'),
 WindowsPath('d:/study-on-agent/dev_skills/tell-joke/references/puns.md')]

In [ ]:
(SKILL_DIR / "read-file" / "main.py").is_file()

True

In [14]:
(SKILL_DIR / "read-file" / "main.py").is_dir()

False

In [12]:
(SKILL_DIR / "read-file" / "main.py").exists()

True

In [17]:
print((SKILL_DIR / "tell-joke" / "references" / "dad-jokes.md").read_text())

# Dad jokes

- I'm reading a book about anti-gravity. It's impossible to put down.
- I used to hate facial hair, but then it grew on me.
- Why don't skeletons fight each other? They don't have the guts.
- I only know 25 letters of the alphabet. I don't know y.
- What do you call a fish with no eyes? A fsh.
- I told my wife she was drawing her eyebrows too high. She looked surprised.
- Why did the scarecrow win an award? He was outstanding in his field.
- I'm on a seafood diet. I see food and I eat it.



In [33]:
def to_argv(args: dict) -> list[str]:
    argv = []
    for key, value in args.items():
        argv += [f"--{key}", str(value)]
    return argv

# dispatcher side: use dev_skills/read-file/main.py directly, now that
# dev_tools is retired -- read the skill's own SKILL.md as a real target
args = to_argv({"file": str(SKILL_DIR / "read-file" / "SKILL.md")})
result = run_script(SKILL_DIR / "read-file", "main.py", args)
# -> subprocess runs: python main.py --file .../SKILL.md
# -> stdout captured, guardrail/exit-code folded in -- that's tool_result
print(result)

---
name: read-file
description: Read a file's contents. Use when the user needs to see what's in a specific file.
---

# Read file

When invoked, do the following:

1. read the file
2. return the content




In [34]:
args

['--file', 'd:\\study-on-agent\\dev_skills\\read-file\\SKILL.md']

In [35]:
import json
from pathlib import Path

def _parse_frontmatter(path: Path) -> dict:
    text = path.read_text(encoding="utf-8")
    _, frontmatter, _body = text.split("---", 2)
    meta = {}
    for line in frontmatter.strip().splitlines():
        key, _, value = line.partition(":")
        meta[key.strip()] = value.strip()
    return meta

In [36]:
_parse_frontmatter(SKILL_DIR/"farewell"/"SKILL.md")

{'name': 'farewell', 'description': 'use when the person say good bye'}

In [37]:
import argparse

def get_args():
    parser = argparse.ArgumentParser(description="test idea")
    parser.add_argument('-f', '--file', required=True, help='File to read', type=str, default="dummy.txt")
    return parser

In [38]:
_ = get_args()

In [39]:
_._actions[1]

_StoreAction(option_strings=['-f', '--file'], dest='file', nargs=None, const=None, default='dummy.txt', type=<class 'str'>, choices=None, required=True, help='File to read', metavar=None)

In [40]:
from dataclasses import dataclass

@dataclass
class Arg:
    field:str
    description:str
    dtype:type[str]|type[int]|type[float]
    required:bool
    default: str|int|float|None=None

@dataclass
class Args:
    args:list[Arg]

In [41]:
arg = _._actions[1]
Arg(
    field=arg.dest,
    description=arg.help,
    dtype=arg.type,
    required=arg.required,
    default=arg.default
)

Arg(field='file', description='File to read', dtype=<class 'str'>, required=True, default='dummy.txt')

In [42]:
_._actions[1].type is str

True

In [43]:
import importlib.util
from pathlib import Path

def load_parser(main_py: Path):
    """Imports main.py by file path (not a package import) and calls its get_args()."""
    spec = importlib.util.spec_from_file_location(main_py.stem, main_py)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)      # runs the module's top level -- but not main(), since it's __main__-guarded
    return module.get_args()  

In [44]:

parser = load_parser(SKILL_DIR/"read-file"/"main.py")
actions = parser._actions[1:]

In [45]:
_args = []

for arg in actions:
    _args.append(
        Arg(
            field=arg.dest,
            description=arg.help,
            dtype=arg.type,
            required=arg.required,
            default=arg.default
        )
    )

args = Args(args=_args)

In [46]:
args

Args(args=[Arg(field='file', description='File to read', dtype=<class 'str'>, required=True, default=None)])

In [47]:
def read_skill(skill_dir: Path) -> dict:
    """Unified skill/tool discovery, keyed by the skill's own folder --
    SKILL.md and main.py are always at fixed names inside it. name+
    description come from SKILL.md's frontmatter; args come from main.py's
    own get_args() parser when a main.py exists (a pure-instruction skill
    like farewell or greeting has none, so args stays empty)."""
    skill_md = skill_dir / "SKILL.md"
    main_py = skill_dir / "main.py"

    meta = _parse_frontmatter(skill_md)

    if main_py.is_file():
        parser = load_parser(main_py)
        args = Args(args=[
            Arg(
                field=action.dest,
                description=action.help or "",
                dtype=action.type if action.type in (str, int, float) else str,
                required=bool(action.required),
                default=action.default,
            )
            for action in parser._actions
            if action.dest != "help"
        ])
    else:
        args = Args(args=[])

    return {
        "name": meta["name"],
        "description": meta["description"],
        "args": args,
        "executable": main_py if main_py.is_file() else None,
        "path": skill_md,
    }

In [48]:
for name in ("farewell", "greeting", "read-file", "tell-joke"):
    print(read_skill(SKILL_DIR / name))

{'name': 'farewell', 'description': 'use when the person say good bye', 'args': Args(args=[]), 'executable': None, 'path': WindowsPath('d:/study-on-agent/dev_skills/farewell/SKILL.md')}
{'name': 'greeting', 'description': 'use when start the conversation', 'args': Args(args=[]), 'executable': None, 'path': WindowsPath('d:/study-on-agent/dev_skills/greeting/SKILL.md')}
{'name': 'read-file', 'description': "Read a file's contents. Use when the user needs to see what's in a specific file.", 'args': Args(args=[Arg(field='file', description='File to read', dtype=<class 'str'>, required=True, default=None)]), 'executable': WindowsPath('d:/study-on-agent/dev_skills/read-file/main.py'), 'path': WindowsPath('d:/study-on-agent/dev_skills/read-file/SKILL.md')}
{'name': 'tell-joke', 'description': 'Tell a joke. Use when the user asks for a joke, wants to be entertained, or needs a laugh.', 'args': Args(args=[]), 'executable': None, 'path': WindowsPath('d:/study-on-agent/dev_skills/tell-joke/SKILL.

## Testing `tell-joke` for real -- brollm + broflow

`tell-joke` has no `main.py` of its own, but its instructions tell the
model to use the `read-file` skill to load whichever reference file fits.
So "available actions" for this run aren't just the invoked skill's own
executable (it has none) -- they're every *other* discoverable skill that
does have one. That's the cross-skill reuse case in practice.

Three steps, same shape as everything built in `dev.ipynb`/`bro_agent`,
just rebuilt here against the new `read_skill`/`Args` shape:
- **plan** -- given `tell-joke`'s instructions + the available actions
  (`read-file`, with its `Args`), ask the model which action (if any) to
  call.
- **act** -- run it for real via `run_script`.
- **respond** -- feed the result back, ask for the final joke.

In [49]:
import boto3
from brollm import BaseContract

BEDROCK_REGION = "us-east-1"
BEDROCK_MODEL_ID = "google.gemma-3-4b-it"
bedrock_client = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)

def call_bedrock(prompt: str) -> str:
    response = bedrock_client.converse(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
    )
    return response["output"]["message"]["content"][0]["text"]


def read_skill_body(skill_md: Path) -> str:
    text = skill_md.read_text(encoding="utf-8")
    _, _frontmatter, body = text.split("---", 2)
    return body.strip()

In [50]:
def describe_action(entry: dict) -> str:
    if not entry["args"].args:
        return f"- {entry['name']}: {entry['description']} (no arguments)"
    arg_list = ", ".join(
        f"{a.field}: {a.dtype.__name__}{'' if a.required else ' (optional)'}"
        for a in entry["args"].args
    )
    return f"- {entry['name']}: {entry['description']} (args: {{{arg_list}}})"


def build_plan_prompt(skill: dict, user_request: str, available_actions: list[dict]) -> str:
    instructions = read_skill_body(skill["path"])
    action_docs = "\n".join(describe_action(a) for a in available_actions) or "(no actions available)"
    return (
        f"{instructions}\n\n"
        f"Available actions:\n{action_docs}\n\n"
        "Decide which action (if any) to call before replying.\n"
        "Reply with only a fenced json codeblock, either:\n"
        '```json\n{"action": "read-file", "args": {"file": "path/to/file.md"}}\n```\n'
        "or, if no action is needed:\n"
        '```json\n{"action": null}\n```\n\n'
        f"User request: {user_request}"
    )


def build_respond_prompt(skill: dict, user_request: str, action: str | None, tool_result: str | None) -> str:
    instructions = read_skill_body(skill["path"])
    prompt = f"{instructions}\n\nUser request: {user_request}\n"
    if action:
        prompt += f"Action `{action}` returned:\n{tool_result}\n\n"
    prompt += "Write the final reply to the user, following the skill's instructions."
    return prompt


def run_action(entry: dict, args: dict) -> str:
    return run_script(entry["path"].parent, "main.py", to_argv(args))

In [51]:
from brollm import extract_codeblocks
from broflow import BaseTask, TaskRegistry, Flow

def _first_json_block(raw: str) -> dict:
    blocks = [b for b in extract_codeblocks(raw) if b.language == "json"]
    return json.loads(blocks[0].content)


class PlanTask(BaseTask):
    possible_next = {"act", "respond"}

    def __call__(self, state, **kwargs):
        prompt = build_plan_prompt(state["skill"], state["user_request"], state["available_actions"])
        raw = call_bedrock(prompt)
        call = _first_json_block(raw)
        state["action"] = call.get("action")
        state["args"] = call.get("args", {})
        self.set_next("act" if state["action"] else "respond")
        return state


class ActTask(BaseTask):
    possible_next = {"respond"}

    def __call__(self, state, **kwargs):
        entry = next(a for a in state["available_actions"] if a["name"] == state["action"])
        state["tool_result"] = run_action(entry, state["args"])
        self.set_next("respond")
        return state


class RespondTask(BaseTask):
    possible_next = {"end"}

    def __call__(self, state, **kwargs):
        prompt = build_respond_prompt(state["skill"], state["user_request"], state.get("action"), state.get("tool_result"))
        state["answer"] = call_bedrock(prompt)
        self.set_next("end")
        return state


_registry = TaskRegistry()
_registry.register("plan", PlanTask("plan"))
_registry.register("act", ActTask("act"))
_registry.register("respond", RespondTask("respond"))
joke_flow = Flow(_registry)

In [52]:
tell_joke = read_skill(SKILL_DIR / "tell-joke")
available_actions = [read_skill(SKILL_DIR / "read-file")]  # every OTHER skill that has a main.py

state = {
    "skill": tell_joke,
    "user_request": "tell me a pun",
    "available_actions": available_actions,
}
joke_flow.run(start="plan", end="end", state=state)

print("trace:", joke_flow.trace)
print("action chosen:", state.get("action"), state.get("args"))
print("tool_result:\n", state.get("tool_result"))
print("\nfinal answer:", state["answer"])

trace: [('plan', 'act'), ('act', 'respond'), ('respond', 'end')]
action chosen: read-file {'file': 'dev_skills/tell-joke/references/puns.md'}
tool_result:
 # Puns

- I used to be a banker, but I lost interest.
- A bicycle can't stand on its own because it's two-tired.
- Time flies like an arrow. Fruit flies like a banana.
- I stayed up all night wondering where the sun went. Then it dawned on me.
- Did you hear about the claustrophobic astronaut? He just needed a little space.
- Velcro: what a rip-off.
- I'm friends with 25 letters of the alphabet. I don't know y.
- A boiled egg every morning is hard to beat.



final answer: Here’s a pun for you: A bicycle can't stand on its own because it’s two-tired.


In [50]:
import boto3
from botocore.exceptions import ClientError

BEDROCK_REGION = "us-east-1"
BEDROCK_MODEL_ID = "google.gemma-3-4b-it"
bedrock_client = boto3.client("bedrock-runtime", region_name=BEDROCK_REGION)


import time

def call_bedrock_stream(prompt: str, delay: float = 0.02) -> str:
    response = bedrock_client.converse_stream(
        modelId=BEDROCK_MODEL_ID,
        messages=[{"role": "user", "content": [{"text": prompt}]}],
    )

    full_text = ""
    for event in response["stream"]:
        if "contentBlockDelta" in event:
            chunk = event["contentBlockDelta"]["delta"].get("text", "")
            for ch in chunk:
                print(ch, end="", flush=True)
                time.sleep(delay)   # purely cosmetic -- the data already arrived, this just paces the display
            full_text += chunk
        elif "metadata" in event:
            usage = event["metadata"].get("usage", {})
            print(f"\n\n[tokens -- in: {usage.get('inputTokens')}, out: {usage.get('outputTokens')}]")

    return full_text
# def call_bedrock_stream(prompt: str) -> str:
#     response = bedrock_client.converse_stream(
#         modelId=BEDROCK_MODEL_ID,
#         messages=[{"role": "user", "content": [{"text": prompt}]}],
#     )

#     full_text = ""
#     for event in response["stream"]:
#         if "contentBlockDelta" in event:
#             chunk = event["contentBlockDelta"]["delta"].get("text", "")
#             print(chunk, end="", flush=True)   # this is the "typing effect"
#             full_text += chunk
#         elif "metadata" in event:
#             usage = event["metadata"].get("usage", {})
#             print(f"\n\n[tokens -- in: {usage.get('inputTokens')}, out: {usage.get('outputTokens')}]")

#     return full_text


try:
    answer = call_bedrock_stream("Tell me a long dad joke.")
except ClientError as e:
    print("Streaming call failed:", e.response["Error"]["Code"], "-", e.response["Error"]["Message"])


Okay, buckle up. This one’s a doozy. It’s a classic, and it requires a little patience…

A man walks into a library and asks for books about paranoia. 

The librarian whispers, “They’re right behind you!”

…

…

…

Okay, okay, I know!  Let me continue! 

The man turns around, sees nothing, and says, "I don't see anything!" 

The librarian whispers again, even more frantically, “They’re *thinking* about you!”

…

…

…

Still not laughing?  Don't worry, it gets to the good part.

The man spins around again, completely bewildered, and says, “I still don’t see anything!”

The librarian, practically shouting now, warns, “They’re *plotting* against you!”

…

…

…

Finally, the man turns around, looks directly at the librarian, and says, “You know what? I think I’ll just take a book about helpfulness.” 

---

How’s that?  Worth the wait? 😄 

Would you like to hear another one, or maybe a different type of joke (like a pun or a one-liner)?

[tokens -- in: 16, out: 265]
